# PAAL Sow Posture Classification — Colab Training

**Workflow:**
1. Code lives on GitHub — edit locally, `git push`, then `!git pull` here
2. Data lives on Google Drive — upload once, mount every session
3. Training runs on Colab GPU

## First-Time Setup
1. Upload `paal_data.zip` to your Google Drive root (see instructions below)
2. Run all cells in order

## Returning Sessions
1. Run Cell 1 (mount Drive)
2. Run Cell 2 (`git pull` to grab latest code)
3. Skip to the training cell you need

---
## 1. Mount Google Drive & Check GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Check GPU
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
## 2. Clone / Update Code from GitHub

In [ ]:
import os

REPO_URL = "https://github.com/AkbarDevop/paal-pipeline.git"
REPO_DIR = "/content/paal_pipeline"

if os.path.exists(REPO_DIR):
    # Already cloned — just pull latest changes
    %cd {REPO_DIR}
    !git pull
else:
    # First time — clone the repo
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

# Install dependencies
!pip install -q timm

print("\nRepo ready!")
!git log --oneline -3

---
## 3. Unzip Data (First Time Only)

**Before running this cell**, upload `paal_data.zip` to your Google Drive root folder.

To create the zip on your Mac:
```bash
cd ~/Downloads/paal_pipeline
zip -r ~/Desktop/paal_data.zip data/ -x '*.raw'
```
This zips only the `.jpg` images (skips `.raw` files to save space).
Then upload `paal_data.zip` from your Desktop to Google Drive.

In [ ]:
import os

DRIVE_ZIP = "/content/drive/MyDrive/paal_data.zip"
DATA_DIR = "/content/paal_pipeline/data"

if os.path.exists(DATA_DIR) and len(os.listdir(DATA_DIR)) > 10:
    print(f"Data already extracted: {len(os.listdir(DATA_DIR))} folders")
else:
    if not os.path.exists(DRIVE_ZIP):
        print(f"ERROR: {DRIVE_ZIP} not found!")
        print("Upload paal_data.zip to your Google Drive root folder first.")
    else:
        print("Extracting data (this takes a few minutes)...")
        !unzip -q {DRIVE_ZIP} -d /content/paal_pipeline/
        print(f"Done! {len(os.listdir(DATA_DIR))} folders extracted")

---
## 4. Verify Setup

In [ ]:
import os
os.chdir("/content/paal_pipeline")

# Check data
data_folders = os.listdir("data")
print(f"Data folders: {len(data_folders)}")

# Check labels
labels = [f for f in os.listdir("labels") if f.endswith('.csv')]
print(f"Label files: {labels}")

# Check model imports work
from models import SingleModalModel, BACKBONES
print(f"Backbones available: {BACKBONES}")

# Check GPU in PyTorch
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training device: {device}")

print("\nAll good! Ready to train.")

---
## 5. Train All 9 Models (3 Backbones x 3 Modalities)

This trains all combinations. On Colab T4 GPU, each model takes ~2-5 min (vs 30-90 min on your Mac).

In [ ]:
import subprocess, itertools

BACKBONES = ["mobilenet_v2", "xception", "densenet121"]
MODALITIES = ["rgb", "ir", "depth"]

for bb, mod in itertools.product(BACKBONES, MODALITIES):
    print(f"\n{'='*60}")
    print(f"  Training: {bb} / {mod}")
    print(f"{'='*60}\n")
    !python train.py \
        --modality {mod} \
        --backbone {bb} \
        --num-classes 3 \
        --labels-csv labels/labels_posture3.csv \
        --model-prefix posture3 \
        --use-cropped \
        --class-weights \
        --epochs 30

print("\n" + "="*60)
print("  ALL TRAINING COMPLETE!")
print("="*60)

---
## 6. Evaluate All Models

In [ ]:
!python eval.py \
    --model-prefix posture3 \
    --class-set posture3 \
    --num-classes 3 \
    --labels-csv labels/labels_posture3.csv \
    --use-cropped

---
## 7. (Optional) Sowbot Pre-training + Fine-tuning

Upload `sowbot_data.zip` to Google Drive first:
```bash
cd ~/Downloads
zip -r ~/Desktop/sowbot_data.zip sowbot/posture_data_2000/
```

In [ ]:
# Unzip sowbot data (first time only)
import os

SOWBOT_ZIP = "/content/drive/MyDrive/sowbot_data.zip"
SOWBOT_DIR = "/content/sowbot/posture_data_2000"

if os.path.exists(SOWBOT_DIR):
    print(f"Sowbot data already extracted")
else:
    if not os.path.exists(SOWBOT_ZIP):
        print(f"Upload sowbot_data.zip to Google Drive first")
    else:
        !unzip -q {SOWBOT_ZIP} -d /content/
        print("Sowbot data extracted")

In [ ]:
# Pre-train all 3 backbones on sowbot
for bb in ["mobilenet_v2", "xception", "densenet121"]:
    print(f"\n{'='*60}")
    print(f"  Pre-training: {bb} on sowbot")
    print(f"{'='*60}\n")
    !python pretrain_sowbot.py --backbone {bb} --epochs 30 --data-root /content/sowbot/posture_data_2000

In [ ]:
# Fine-tune sowbot-pretrained models on OAK data
import itertools

BACKBONES = ["mobilenet_v2", "xception", "densenet121"]
MODALITIES = ["rgb", "ir", "depth"]

for bb, mod in itertools.product(BACKBONES, MODALITIES):
    print(f"\n{'='*60}")
    print(f"  Fine-tuning: {bb} / {mod} (sowbot pre-trained)")
    print(f"{'='*60}\n")
    !python train.py \
        --modality {mod} \
        --backbone {bb} \
        --init-weights models/sowbot_{bb}_best.pth \
        --num-classes 3 \
        --labels-csv labels/labels_posture3.csv \
        --model-prefix posture3ft \
        --use-cropped \
        --class-weights \
        --lr 1e-5 \
        --epochs 30

---
## 8. Save Models Back to Google Drive

Copy trained models and outputs to Drive so they persist across sessions.

In [ ]:
import shutil, os

DRIVE_SAVE = "/content/drive/MyDrive/paal_results"
os.makedirs(DRIVE_SAVE, exist_ok=True)

# Copy models
if os.path.exists("models"):
    shutil.copytree("models", f"{DRIVE_SAVE}/models", dirs_exist_ok=True)
    print(f"Models saved to Drive: {len(os.listdir('models'))} files")

# Copy outputs (plots, eval results)
if os.path.exists("outputs"):
    shutil.copytree("outputs", f"{DRIVE_SAVE}/outputs", dirs_exist_ok=True)
    print(f"Outputs saved to Drive: {len(os.listdir('outputs'))} files")

print(f"\nAll results saved to: {DRIVE_SAVE}")

---
## 9. (Optional) Load Previous Models from Drive

If you trained before and want to resume or evaluate:

In [ ]:
import shutil, os

DRIVE_SAVE = "/content/drive/MyDrive/paal_results"

if os.path.exists(f"{DRIVE_SAVE}/models"):
    os.makedirs("models", exist_ok=True)
    shutil.copytree(f"{DRIVE_SAVE}/models", "models", dirs_exist_ok=True)
    print(f"Loaded {len(os.listdir('models'))} model files from Drive")
else:
    print("No saved models found on Drive")